# Batch 06: SST components, source flags and uncertainties
Read-only development review of 21 columns. No temperature imputation, clipping, source-record deletion, new quality threshold or test access.
Absolute SST conversion subtracts 273.15; a diurnal anomaly is an increment and receives no offset. Existing source-specific flags are retained separately from model-input choices.

In [1]:
from pathlib import Path
import json,hashlib,collections
import numpy as np
import pyarrow.parquet as pq
from data3001.dataset import development_ids,track_path
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/dataset_manifest.json').exists())
manifest=json.loads((root/'configs/dataset_manifest.json').read_text())
assert hashlib.sha256((root/'docs/contract/data_contract_v1.0.xlsx').read_bytes()).hexdigest()==manifest['contract_sha256']
base=['sst','sst1','sst2'];fields=[]
for k in base:fields.extend([k,'flg_'+k,k+'_ok',k+'_celsius' if k!='sst2' else 'sst2_anomaly_K','err_'+k,'err_'+k+'_ok','err_'+k+'_negative_unresolved'])
nulls={k:0 for k in fields};checks={k:0 for k in ['infinite_values','invalid_flag_codes','converted_value_mismatch','sst_qualification_mismatch','uncertainty_flag_mismatch','negative_flag_mismatch','raw_temperature_missing_masks_disagree']}
metrics={scope:{k:{'rows':0,'raw_missing':0,'raw_finite_but_unusable':0,'usable':0,'usable_negative_values':0,'usable_min':None,'usable_max':None,'error_missing':0,'error_negative':0,'flag_counts':{str(i):0 for i in range(6)},'raw_missing_IDs':set(),'IDs_all_raw_missing':set()} for k in base} for scope in ['development_all','R_observations']}
seasonal={str(i):{'R_rows':0,'sst_usable':0,'sst1_usable':0,'sst2_usable':0} for i in range(4)}
rows=0;sum_checks={'finite_triplets':0,'max_abs_residual_K':0.0,'residual_above_0_001_K':0};examples=[]
for ident in development_ids():
    tab=pq.read_table(track_path(ident),columns=fields+['in_R','time','source_obs','season'])
    a={k:tab[k].to_numpy() for k in tab.column_names};n=tab.num_rows;rows+=n
    for k in fields:nulls[k]+=tab[k].null_count
    masks=[np.isnan(a[k]) for k in base]
    checks['raw_temperature_missing_masks_disagree']+=int(((masks[0]!=masks[1])|(masks[0]!=masks[2])).sum())
    for k in base:
        raw=a[k].astype(float);flag=a['flg_'+k];converted=raw-(0 if k=='sst2' else 273.15);stored=a[k+'_celsius' if k!='sst2' else 'sst2_anomaly_K']
        valid=np.isfinite(raw)&(flag==5) if k=='sst2' else np.isfinite(raw)&np.isin(flag,[4,5])&(converted>=-2)&(converted<=50)
        checks['invalid_flag_codes']+=int((~np.isin(flag,np.arange(6))).sum());checks['converted_value_mismatch']+=int((~np.isclose(stored,converted,equal_nan=True,rtol=0,atol=1e-10)).sum());checks['sst_qualification_mismatch']+=int((a[k+'_ok']!=valid).sum())
        err=a['err_'+k];checks['uncertainty_flag_mismatch']+=int((a['err_'+k+'_ok']!=(np.isfinite(err)&(err>=0))).sum());checks['negative_flag_mismatch']+=int((a['err_'+k+'_negative_unresolved']!=(np.isfinite(err)&(err<0))).sum())
        checks['infinite_values']+=int(np.isinf(raw).sum()+np.isinf(err).sum())
        for scope,mask in [('development_all',np.ones(n,dtype=bool)),('R_observations',a['in_R'])]:
            z=metrics[scope][k];z['rows']+=int(mask.sum());missing=np.isnan(raw)
            z['raw_missing']+=int((mask&missing).sum());z['raw_finite_but_unusable']+=int((mask&np.isfinite(raw)&~valid).sum());z['usable']+=int((mask&valid).sum());z['usable_negative_values']+=int((mask&valid&(converted<0)).sum())
            z['error_missing']+=int((mask&np.isnan(err)).sum());z['error_negative']+=int((mask&np.isfinite(err)&(err<0)).sum())
            if (mask&missing).any():z['raw_missing_IDs'].add(ident)
            if mask.any() and missing[mask].all():z['IDs_all_raw_missing'].add(ident)
            for i in range(6):z['flag_counts'][str(i)]+=int((mask&(flag==i)).sum())
            vals=converted[mask&valid]
            if vals.size:
                z['usable_min']=float(vals.min()) if z['usable_min'] is None else min(z['usable_min'],float(vals.min()))
                z['usable_max']=float(vals.max()) if z['usable_max'] is None else max(z['usable_max'],float(vals.max()))
        if k=='sst' and len(examples)<5:
            idx=np.flatnonzero(a['in_R']&np.isfinite(raw)&~valid)
            if idx.size:
                j=int(idx[0]);examples.append({'id':ident,'source_obs':int(a['source_obs'][j]),'UTC':str(np.datetime64(int(a['time'][j]),'s')),'sst_celsius_unfiltered':float(converted[j]),'flg_sst':int(flag[j]),'sst_ok':bool(a['sst_ok'][j])})
    finite=np.isfinite(a['sst'])&np.isfinite(a['sst1'])&np.isfinite(a['sst2'])
    residual=np.abs(a['sst'][finite].astype(float)-a['sst1'][finite].astype(float)-a['sst2'][finite].astype(float))
    sum_checks['finite_triplets']+=int(finite.sum())
    if residual.size:sum_checks['max_abs_residual_K']=max(sum_checks['max_abs_residual_K'],float(residual.max()));sum_checks['residual_above_0_001_K']+=int((residual>0.001).sum())
    for i in range(4):
        mask=a['in_R']&(a['season']==i);z=seasonal[str(i)];z['R_rows']+=int(mask.sum())
        for k in base:z[k+'_usable']+=int((mask&a[k+'_ok']).sum())
for scope in metrics.values():
    for z in scope.values():
        for key in ['raw_missing_IDs','IDs_all_raw_missing']:z[key]=len(z[key])
        z['raw_missing_pct']=100*z['raw_missing']/z['rows'];z['unusable']=z['rows']-z['usable'];z['unusable_pct']=100*z['unusable']/z['rows']
assert rows==33943292 and not any(checks.values())
result={'scope':'Development only, final test not opened. Usable means passing the existing frozen SST predicates, not verified physical truth or predictive usefulness.','IDs':len(development_ids()),'rows':rows,'release':manifest['version'],'contract_sha256':manifest['contract_sha256'],'null_rows_by_field':nulls,'consistency_failures':checks,'metrics':metrics,'R_seasonal_observation_counts':seasonal,'sum_sst1_sst2_check':sum_checks,'unfiltered_finite_R_examples':examples,'input_file_checksums_verified':len(development_ids()),'data_modified':False,'imputation_performed':False,'new_temperature_thresholds_selected':False}
print(json.dumps(result,indent=2))


{
  "scope": "Development only, final test not opened. Usable means passing the existing frozen SST predicates, not verified physical truth or predictive usefulness.",
  "IDs": 3224,
  "rows": 33943292,
  "release": "per-drifter-20261016-v1",
  "contract_sha256": "6cfa91471a1399a9c3a1a8edfeb2e18cea17e0f4031ef0f8ca604ca4565a2446",
  "null_rows_by_field": {
    "sst": 1194141,
    "flg_sst": 0,
    "sst_ok": 0,
    "sst_celsius": 1194141,
    "err_sst": 1195051,
    "err_sst_ok": 0,
    "err_sst_negative_unresolved": 0,
    "sst1": 1194141,
    "flg_sst1": 0,
    "sst1_ok": 0,
    "sst1_celsius": 1194141,
    "err_sst1": 1195051,
    "err_sst1_ok": 0,
    "err_sst1_negative_unresolved": 0,
    "sst2": 1194141,
    "flg_sst2": 0,
    "sst2_ok": 0,
    "sst2_anomaly_K": 1194141,
    "err_sst2": 1195051,
    "err_sst2_ok": 0,
    "err_sst2_negative_unresolved": 0
  },
  "consistency_failures": {
    "infinite_values": 0,
    "invalid_flag_codes": 0,
    "converted_value_mismatch": 0,
    "s

## Additional source-identity finding
The original numeric/flag recomputation checks do not validate the physical values. A new check below stratifies departures from the documented component-sum identity; its results must be considered before modelling with these components. No source values or qualification rules were changed.

In [2]:
import json
import numpy as np
import pyarrow.parquet as pq
from data3001.dataset import development_ids,track_path
thresholds=[0.005,0.01,0.1,1,100,200]
counts={scope:{'rows':0,**{str(t):0 for t in thresholds},'max_abs_residual_K':0.0} for scope in ['finite_all','qualified_all','all_flag5','R_finite','R_qualified','R_all_flag5']}
top={scope:[] for scope in ['qualified_all','R_qualified']}
for ident in development_ids():
    tab=pq.read_table(track_path(ident),columns=['sst','sst1','sst2','sst_ok','sst1_ok','sst2_ok','flg_sst','flg_sst1','flg_sst2','in_R','time','source_obs'])
    a={k:tab[k].to_numpy() for k in tab.column_names}
    finite=np.isfinite(a['sst'])&np.isfinite(a['sst1'])&np.isfinite(a['sst2'])
    qualified=finite&a['sst_ok']&a['sst1_ok']&a['sst2_ok']
    fives=finite&(a['flg_sst']==5)&(a['flg_sst1']==5)&(a['flg_sst2']==5)
    residual=np.abs(a['sst'].astype(float)-a['sst1'].astype(float)-a['sst2'].astype(float))
    masks={'finite_all':finite,'qualified_all':qualified,'all_flag5':fives,'R_finite':finite&a['in_R'],'R_qualified':qualified&a['in_R'],'R_all_flag5':fives&a['in_R']}
    for scope,mask in masks.items():
        c=counts[scope];c['rows']+=int(mask.sum())
        if mask.any():c['max_abs_residual_K']=max(c['max_abs_residual_K'],float(residual[mask].max()))
        for t in thresholds:c[str(t)]+=int((mask&(residual>t)).sum())
        if scope in top and mask.any():
            idx=np.flatnonzero(mask);idx=idx[np.argsort(residual[idx])[-2:]]
            for j in idx:
                top[scope].append({'id':ident,'source_obs':int(a['source_obs'][j]),'UTC':str(np.datetime64(int(a['time'][j]),'s')),'sst_K':float(a['sst'][j]),'sst1_K':float(a['sst1'][j]),'sst2_anomaly_K':float(a['sst2'][j]),'flg_sst':int(a['flg_sst'][j]),'flg_sst1':int(a['flg_sst1'][j]),'flg_sst2':int(a['flg_sst2'][j]),'abs_residual_K':float(residual[j])})
            top[scope]=sorted(top[scope],key=lambda r:r['abs_residual_K'],reverse=True)[:4]
result_identity={'definition':'Absolute residual abs(sst - sst1 - sst2); thresholds describe source-identity discrepancies, not newly selected temperature-deletion rules.','counts':counts,'top_examples':top,'data_modified':False,'cause_established':False}
print(json.dumps(result_identity,indent=2))


{
  "definition": "Absolute residual abs(sst - sst1 - sst2); thresholds describe source-identity discrepancies, not newly selected temperature-deletion rules.",
  "counts": {
    "finite_all": {
      "rows": 32749151,
      "0.005": 44,
      "0.01": 44,
      "0.1": 44,
      "1": 44,
      "100": 44,
      "200": 44,
      "max_abs_residual_K": 273.15101623535156
    },
    "qualified_all": {
      "rows": 25578579,
      "0.005": 0,
      "0.01": 0,
      "0.1": 0,
      "1": 0,
      "100": 0,
      "200": 0,
      "max_abs_residual_K": 0.0010303258895874023
    },
    "all_flag5": {
      "rows": 25578266,
      "0.005": 0,
      "0.01": 0,
      "0.1": 0,
      "1": 0,
      "100": 0,
      "200": 0,
      "max_abs_residual_K": 0.0010303258895874023
    },
    "R_finite": {
      "rows": 302705,
      "0.005": 0,
      "0.01": 0,
      "0.1": 0,
      "1": 0,
      "100": 0,
      "200": 0,
      "max_abs_residual_K": 0.0010302737355232239
    },
    "R_qualified": {
      "rows

## Retained limitation
All 44 large sum-identity discrepancies are outside R and fail total-SST screening. Existing SST rules nevertheless admit approximately 49 C R observations, including a source flag-5 example. Passing a deterministic predicate is not scientific confirmation of regional plausibility. Source/context review is required before using SST in an enhanced model; this does not invalidate the core position-only workflow.

In [3]:
import duckdb,json
c=duckdb.connect();c.execute('SET threads=2')
q="""SELECT count(*) AS rows, count(DISTINCT id) AS IDs,
sum(CASE WHEN sst_ok THEN 1 ELSE 0 END) AS total_sst_pass,
sum(CASE WHEN sst1_ok THEN 1 ELSE 0 END) AS non_diurnal_pass,
sum(CASE WHEN sst2_ok THEN 1 ELSE 0 END) AS diurnal_pass,
sum(CASE WHEN in_R THEN 1 ELSE 0 END) AS R_rows
FROM read_parquet('data/train/*.parquet')
WHERE abs(CAST(sst AS DOUBLE)-CAST(sst1 AS DOUBLE)-CAST(sst2 AS DOUBLE))>0.01"""
r=c.execute(q).fetchdf().to_dict('records')[0]
print('IDENTITY_OUTLIER_DETAIL',json.dumps(r))
q2="""SELECT id, time_UTC, sst_celsius, flg_sst, sst1_celsius, flg_sst1, sst2_anomaly_K, flg_sst2
FROM read_parquet('data/train/*.parquet') WHERE in_R AND sst_ok
ORDER BY sst_celsius DESC LIMIT 2"""
print('HIGH_R_SST_PASS_EXISTING_RULE',c.execute(q2).fetchdf().to_json(orient='records',date_format='iso'))


IDENTITY_OUTLIER_DETAIL {"rows": 44, "IDs": 4, "total_sst_pass": 0.0, "non_diurnal_pass": 7.0, "diurnal_pass": 0.0, "R_rows": 0.0}
HIGH_R_SST_PASS_EXISTING_RULE [{"id":43869,"time_UTC":"2014-10-13T00:00:00.000Z","sst_celsius":49.1270080566,"flg_sst":4,"sst1_celsius":-0.4679931641,"flg_sst1":4,"sst2_anomaly_K":49.5960006714,"flg_sst2":3},{"id":43869,"time_UTC":"2014-09-09T09:00:00.000Z","sst_celsius":49.0880065918,"flg_sst":5,"sst1_celsius":21.9150024414,"flg_sst1":5,"sst2_anomaly_K":27.1730003357,"flg_sst2":4}]


## Missing-temperature gap profile for imputation planning
Read-only analysis of development pilot visitors. Bracketing by existing SST gates does not approve anchor reliability, retrospective interpolation for live forecasting, or any specific gap limit. No values are filled.

In [4]:
from pathlib import Path
import json,csv
import numpy as np
import pyarrow.parquet as pq
from data3001.dataset import manifest,track_path
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/dataset_index.csv').exists())
m=manifest()
with (root/'configs/dataset_index.csv').open(newline='') as f:
    IDs=sorted(int(r['id']) for r in csv.DictReader(f) if r['pool']=='train' and r['in_R']=='True')
assert all(m['records'][str(i)]['pool']=='train' for i in IDs)
# Complete trajectories avoid artificially splitting gaps at the R boundary.
thresholds=[1,3,6,24,72]
stats={kind:{'R_missing_rows':0,'runs_intersecting_R':0,'R_missing_rows_in_closed_gaps':0,'R_missing_rows_in_closed_gaps_le_hours':{str(k):0 for k in thresholds},'max_consecutive_missing_hours':0,'IDs_with_R_missing':set(),'IDs_all_trajectory_unavailable':set()} for kind in ['raw_missing','unavailable_by_existing_SST_rule']}
examples=[];Rrows=0
for ident in IDs:
    tab=pq.read_table(track_path(ident),columns=['time','sst','sst_ok','in_R','source_obs'])
    t=tab['time'].to_numpy();sst=tab['sst'].to_numpy();ok=tab['sst_ok'].to_numpy();R=tab['in_R'].to_numpy();n=len(t);Rrows+=int(R.sum())
    step=np.diff(t)==3600
    for kind,bad in [('raw_missing',~np.isfinite(sst)),('unavailable_by_existing_SST_rule',~ok)]:
        z=stats[kind];z['R_missing_rows']+=int((bad&R).sum())
        if (bad&R).any():z['IDs_with_R_missing'].add(ident)
        if bad.all():z['IDs_all_trajectory_unavailable'].add(ident)
        starts=np.flatnonzero(bad&~np.r_[False,bad[:-1]&step]);ends=np.flatnonzero(bad&~np.r_[bad[1:]&step,False])
        assert len(starts)==len(ends)
        for lo,hi in zip(starts,ends):
            nr=int(R[lo:hi+1].sum())
            if nr==0:continue
            length=int(hi-lo+1);z['runs_intersecting_R']+=1;z['max_consecutive_missing_hours']=max(z['max_consecutive_missing_hours'],length)
            left=lo>0 and step[lo-1] and ok[lo-1]
            right=hi<n-1 and step[hi] and ok[hi+1]
            closed=bool(left and right)
            if closed:
                z['R_missing_rows_in_closed_gaps']+=nr
                for th in thresholds:
                    if length<=th:z['R_missing_rows_in_closed_gaps_le_hours'][str(th)]+=nr
            if len(examples)<6 and kind=='raw_missing':examples.append({'id':ident,'start_UTC':str(np.datetime64(int(t[lo]),'s')),'end_UTC':str(np.datetime64(int(t[hi]),'s')),'consecutive_missing_rows':length,'missing_rows_inside_R':nr,'bracketed_by_existing_rule_SST':closed})
for z in stats.values():
    for k in ['IDs_with_R_missing','IDs_all_trajectory_unavailable']:z[k]=len(z[k])
    z['R_missing_pct']=100*z['R_missing_rows']/Rrows
assert len(IDs)==396 and Rrows==318108 and stats['raw_missing']['R_missing_rows']==15403
result={'scope':'Development IDs visiting R only, read as complete trajectories; no final-test trajectories, external SST downloads or filled values.','pilot_IDs':len(IDs),'R_rows':Rrows,'stats':stats,'examples':examples,'definitions':{'closed_gap':'Immediately preceding and following stored observations exist exactly one hour from each gap boundary and pass the existing SST predicate. This does not prove physical anchor reliability or forecast-time availability.','gap_length':'Number of consecutive unavailable hourly records, not endpoint-to-endpoint bracket duration. Timestamp discontinuities split runs.','thresholds':'1/3/6/24/72 missing-hour lengths are descriptive bins, not approved imputation limits.'},'imputation_performed':False}
print(json.dumps(result,indent=2))


{
  "scope": "Development IDs visiting R only, read as complete trajectories; no final-test trajectories, external SST downloads or filled values.",
  "pilot_IDs": 396,
  "R_rows": 318108,
  "stats": {
    "raw_missing": {
      "R_missing_rows": 15403,
      "runs_intersecting_R": 69,
      "R_missing_rows_in_closed_gaps": 1141,
      "R_missing_rows_in_closed_gaps_le_hours": {
        "1": 0,
        "3": 3,
        "6": 3,
        "24": 3,
        "72": 3
      },
      "max_consecutive_missing_hours": 18602,
      "IDs_with_R_missing": 61,
      "IDs_all_trajectory_unavailable": 7,
      "R_missing_pct": 4.842066216505087
    },
    "unavailable_by_existing_SST_rule": {
      "R_missing_rows": 15485,
      "runs_intersecting_R": 101,
      "R_missing_rows_in_closed_gaps": 1225,
      "R_missing_rows_in_closed_gaps_le_hours": {
        "1": 8,
        "3": 56,
        "6": 87,
        "24": 87,
        "72": 87
      },
      "max_consecutive_missing_hours": 18602,
      "IDs_with_R